# Pipeline Smoke Test

This notebook imports the same package functions used by CLI runs. It is intentionally thin so research logic stays in `src/ablation_study_jepa/`.

In [1]:
from pathlib import Path
import pandas as pd
from ablation_study_jepa.data.synthetic import build_sample_panel
from ablation_study_jepa.config.loader import load_config
from ablation_study_jepa.builders.data import build_data
from ablation_study_jepa.builders.datasets import build_datasets

root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
# config = load_config(root / 'configs/exp/jepa_ablation.yaml')
config = load_config(root / 'configs/exp/tft.yaml')
(root / 'data/prices').mkdir(parents=True, exist_ok=True)
data_end = config.splits.test_end or config.data.end_date
periods = len(pd.bdate_range(start=config.data.start_date, end=data_end))
build_sample_panel(start=config.data.start_date, periods=periods).to_csv(root / 'data/prices/panel.csv', index=False)
config.data.data_dir = root / 'data/prices'
config.data.macro_data_path = root / 'data/macro/fred_md/fred_md_1960_2025.csv'
prepared = build_data(config)
datasets = build_datasets(config, prepared.scaled_panel, prepared.splits)
len(datasets.train), len(datasets.val), len(datasets.test), datasets.train[0]['x'].shape

(7712, 2292, 1148, torch.Size([60, 24]))

In [ ]:
from ablation_study_jepa.api.experiment import ExperimentRunner

experiment_config = load_config(root / 'configs/exp/tft.yaml')
experiment_config.data.data_dir = root / 'data/prices'
experiment_config.data.macro_data_path = root / 'data/macro/fred_md/fred_md_1960_2025.csv'
experiment_config.evaluation.predictions_dir = root / 'predictions'

result = ExperimentRunner(experiment_config).run()
result


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
/Users/henrrb/Desktop/NTNU/Research/jepa-financial-time-series-ablation/.venv/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type                 ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model           │ TFTWithJEPA          │  1.6 M │ train │     0 │
│ 1 │ criterion       │ MSELoss              │      0 │ train │     0 │
│ 2 │ jepa_module     │ MultiLayerJEPAModule │ 99.5 K │ train │     0 │
│ 3 │ loss_aggregator │ LossAggregator       │      0 │ train │     0 │
└───┴─────────────────┴──────────────────────┴────────┴───────┴───────┘

Trainable params: 1.6 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.6 M                                                                                                
Total estimated model params size (MB): 6                                                                          
Modules in train mode: 128                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/Users/henrrb/Desktop/NTNU/Research/jepa-financial-time-series-ablation/.venv/lib/python3.11/site-packages/lightnin
g/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, 
TreeSpec) and treespec.is_leaf()` instead.

Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│   test/prediction_loss    │   0.0003874183166772127   │
└───────────────────────────┴───────────────────────────┘

{
  "test": {
    "correlation": -0.01603371855256675,
    "directional_accuracy": 0.509581881533101,
    "mae": 0.015315228656378225,
    "mse": 0.00038741831281836995,
    "spearman_rank_ic": -0.006742545472560079,
    "top_bottom_quantile_spread": -1.8015477678051434e-05
  },
  "val": {
    "correlation": -0.030880276484021578,
    "directional_accuracy": 0.48472949389179754,
    "mae": 0.01661311862090749,
    "mse": 0.0004442139505173511,
    "spearman_rank_ic": -0.024130016830935335,
    "top_bottom_quantile_spread": -0.001872641139751321
  }
}


ExperimentResult(run_name='tft_jepa_last1_h1_lambda005', val_metrics={'mse': 0.0004442139505173511, 'mae': 0.01661311862090749, 'correlation': -0.030880276484021578, 'directional_accuracy': 0.48472949389179754, 'spearman_rank_ic': -0.024130016830935335, 'top_bottom_quantile_spread': -0.001872641139751321}, test_metrics={'mse': 0.00038741831281836995, 'mae': 0.015315228656378225, 'correlation': -0.01603371855256675, 'directional_accuracy': 0.509581881533101, 'spearman_rank_ic': -0.006742545472560079, 'top_bottom_quantile_spread': -1.8015477678051434e-05}, prediction_paths={'val': PosixPath('/Users/henrrb/Desktop/NTNU/Research/jepa-financial-time-series-ablation/predictions/tft_jepa_last1_h1_lambda005_jepa_last_L_horizon_1_lambda_0_05_20260509T222536Z_23e287527b/val.csv'), 'test': PosixPath('/Users/henrrb/Desktop/NTNU/Research/jepa-financial-time-series-ablation/predictions/tft_jepa_last1_h1_lambda005_jepa_last_L_horizon_1_lambda_0_05_20260509T222536Z_23e287527b/test.csv')}, metrics_path